In [1]:
import pandas as pd
import requests
import time

In [2]:
rating = pd.read_csv("full_data/rating.csv")
movie = pd.read_csv("full_data/movie.csv")
tag = pd.read_csv("full_data/tag.csv")
link = pd.read_csv("full_data/link.csv")
genome_tags = pd.read_csv("full_data/genome_tags.csv")
genome_scores = pd.read_csv("full_data/genome_scores.csv")

In [3]:
tag_agg = tag.groupby(['userId', 'movieId'])['tag'].apply(list).reset_index()
tag_agg.rename(columns={'tag': 'tags'}, inplace=True)

In [4]:
genome = (genome_scores
             .merge(genome_tags, on='tagId', how='left')
             .sort_values('relevance', ascending=False)
             .groupby('movieId')
             .head(10)
             .groupby('movieId')
             .apply(lambda x: dict(zip(x['tag'], x['relevance'])))
             .reset_index()
             .rename(columns={0: 'genome_tags'})
)


In [5]:
df = (rating
      .merge(movie, on='movieId', how='left')
      .merge(link[['movieId', 'imdbId', 'tmdbId']], on='movieId', how='left')
      .merge(tag_agg, on=['userId', 'movieId'], how='left')
      .merge(genome, on='movieId', how='left')
)

df['genres'] = df['genres'].str.split('|')

In [6]:
TMDB_API_KEY = "15abd04b80ec2a385edbe69a4ebe5451"

In [7]:
def get_tmdb_metadata(tmdb_id):
    base_url = f"https://api.themoviedb.org/3/movie/{tmdb_id}"
    params = {"api_key": TMDB_API_KEY}

    # existing metadata
    r = requests.get(base_url, params=params)
    if r.status_code != 200:
        return {}
    d = r.json()

    # watch providers
    wp = requests.get(base_url + "/watch/providers", params=params)
    streaming = []
    if wp.status_code == 200:
        results = wp.json().get("results", {})
        us = results.get("US", {})  # change country code as needed
        streaming = [p["provider_name"] for p in us.get("flatrate", [])]

    return {
        "overview":           d.get("overview"),
        "popularity":         d.get("popularity"),
        "vote_average":       d.get("vote_average"),
        "vote_count":         d.get("vote_count"),
        "release_date":       d.get("release_date"),
        "runtime":            d.get("runtime"),
        "budget":             d.get("budget"),
        "revenue":            d.get("revenue"),
        "original_language":  d.get("original_language"),
        "production_countries": [c["name"] for c in d.get("production_countries", [])],
        "spoken_languages":   [l["name"] for l in d.get("spoken_languages", [])],
        "streaming":          streaming,  # e.g. ["Netflix", "Hulu"]
        "poster_url":         f"https://image.tmdb.org/t/p/w500{d['poster_path']}" if d.get("poster_path") else None,
    }

In [8]:
unique_movies = df.drop_duplicates('tmdbId')[['movieId', 'tmdbId']].dropna()

results = []
for _, row in unique_movies.iterrows():
    meta = get_tmdb_metadata(int(row['tmdbId']))
    meta['movieId'] = row['movieId']
    results.append(meta)
    time.sleep(0.25)  # rate limit: ~4 req/sec

tmdb = pd.DataFrame(results)

In [9]:
df = df.merge(tmdb, on='movieId', how='left')

In [12]:
df.to_csv("df.csv")

In [13]:
df

,userId,movieId,rating,timestamp,title,genres,imdbId,tmdbId,tags,genome_tags,...,vote_count,release_date,runtime,budget,revenue,original_language,production_countries,spoken_languages,streaming,poster_url
0,1,2,3.5,2005-04-02 23:53:47,Jumanji (1995),"[Adventure, Children, Fantasy]",113497,8844.0,NaN,"{'adventure': 0.981, 'jungle': 0.967, 'childre...",...,11355.0,1995-12-15,104.0,65000000.0,262821940.0,en,[United States of America],"[English, Français]","[Netflix, Philo, Netflix Standard with Ads]",https://image.tmdb.org/t/p/w500/iWV47r6kFneCiA...
1,1,29,3.5,2005-04-02 23:31:16,"City of Lost Children, The (Cité des enfants p...","[Adventure, Drama, Fantasy, Mystery, Sci-Fi]",112682,902.0,NaN,"{'dark fantasy': 0.998, 'visually stunning': 0...",...,1221.0,1995-05-17,112.0,18000000.0,1738611.0,fr,"[France, Germany, Spain, United Kingdom]",[Français],[],https://image.tmdb.org/t/p/w500/whwT3Q9JxbAYzE...
2,1,32,3.5,2005-04-02 23:33:39,Twelve Monkeys (a.k.a. 12 Monkeys) (1995),"[Mystery, Sci-Fi, Thriller]",114746,63.0,NaN,"{'future': 0.99825, 'time loop': 0.99625, 'tim...",...,9167.0,1995-12-29,129.0,29000000.0,168841459.0,en,[United States of America],"[English, Français]",[],https://image.tmdb.org/t/p/w500/gt3iyguaCIw8Dp...
3,1,47,3.5,2005-04-02 23:32:07,Seven (a.k.a. Se7en) (1995),"[Mystery, Thriller]",114369,807.0,NaN,"{'serial killer': 0.99675, 'detective': 0.9852...",...,23221.0,1995-09-22,127.0,33000000.0,327311859.0,en,[United States of America],[English],[],https://image.tmdb.org/t/p/w500/191nKfP0ehp3uI...
4,1,50,3.5,2005-04-02 23:29:40,"Usual Suspects, The (1995)","[Crime, Mystery, Thriller]",114814,629.0,NaN,"{'oscar (best supporting actor)': 0.99875, 'su...",...,11470.0,1995-07-19,106.0,6000000.0,23300000.0,en,[United States of America],"[Español, English, Français, Magyar]",[YouTube TV],https://image.tmdb.org/t/p/w500/99X2SgyFunJFXG...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20000258,138493,68954,4.5,2009-11-13 15:42:00,Up (2009),"[Adventure, Animation, Children, Drama]",1049413,14160.0,NaN,"{'pixar animation': 0.99475, 'computer animati...",...,21742.0,2009-05-28,96.0,175000000.0,735103954.0,en,[United States of America],[English],[Disney Plus],https://image.tmdb.org/t/p/w500/mFvoEwSfLqbcWw...
20000259,138493,69526,4.5,2009-12-03 18:31:48,Transformers: Revenge of the Fallen (2009),"[Action, Adventure, Sci-Fi, IMAX]",1055369,8373.0,NaN,"{'giant robots': 0.9985, 'robots': 0.99775, 'a...",...,9220.0,2009-06-19,150.0,200000000.0,836304167.0,en,[United States of America],"[Español, English]","[Paramount Plus Premium, Paramount Plus Essent...",https://image.tmdb.org/t/p/w500/pLBb0whOzVDtJv...
20000260,138493,69644,3.0,2009-12-07 18:10:57,Ice Age: Dawn of the Dinosaurs (2009),"[Action, Adventure, Animation, Children, Comed...",1080016,8355.0,NaN,"{'animation': 0.985, 'talking animals': 0.981,...",...,8904.0,2009-06-26,94.0,90000000.0,886686817.0,en,[United States of America],[English],[Disney Plus],https://image.tmdb.org/t/p/w500/cXOLaxcNjNAYmE...
20000261,138493,70286,5.0,2009-11-13 15:42:24,District 9 (2009),"[Mystery, Sci-Fi, Thriller]",1136608,17654.0,NaN,"{'alien': 0.999, 'aliens': 0.99225, 'allegory'...",...,10391.0,2009-08-05,112.0,30000000.0,210888950.0,en,"[New Zealand, South Africa, United States of A...","[Afrikaans, , , , isiZulu, English]",[],https://image.tmdb.org/t/p/w500/tuGlQkqLxnodDS...
